In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR

In [2]:
sim = SIMULATOR()

# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/mmul/'
kernel_number = 1 
column_usage = [True, True] 
nInstrPerCol = 47 
imem_add_start = 0 
srf_spm_addres = 0 
version="_bb16x16_2col"

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [3]:
# --------------------------------------------
#                DATA SIZES
# --------------------------------------------
# DISCO-CGRA Configuration
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 2

# Basic Block
BB_ROWS_A = 8
BB_COLS_A = 16
BB_ROWS_B = 16
BB_COLS_B = 8
# C 16x8 to fit VWR
BB_ROWS_C = 16
BB_COLS_C = 8

In [4]:
# Our test
nRowsA = 16
nColsA = 16
nRowsB = nColsA
nColsB = 16

#matrix_A = np.random.randint(1, 15, size=(nRowsA*nColsA))
#matrix_B = np.random.randint(1, 15, size=(nColsA*nColsB))
matrix_A = np.array([i for i in range(nRowsA*nColsA)])
matrix_B = np.array([i for i in range(nRowsB*nColsB)])
matrix_C = np.zeros((nRowsA*nColsB), dtype=int)

In [5]:
# --------------------------------------------
#                LOAD SPM DATA
# --------------------------------------------
# SPM[0] = SRF
# SPM[1] = A00, SPM[2] = A10
# SPM[3] = B00, SPM[4] = B01
# SPM[5] = C00, SPM[6] = C10
# --------------------------------------------
# SRF[0] = nItLoop1 = 16 = n elems of the same row of A per RC 
# SRF[1] = Line of the SPM where the block of A is stored
# SRF[2] = Line of the SPM where the block of B is stored
# SRF[3] = Line of the SPM where the block of C is stored
# --------------------------------------------

# Default SPM lines
srf_spm_line = 0
a_spm_line = 1
b_spm_line = 3
c_spm_line = 5


# Default SRF values
srf = [0 for i in range(SPM_NWORDS)]
srf[0] = 16     # nIt (same for both cols)
srf[1] = 1      # SPM for A (same for both cols)
srf[2] = 3      # SPM for B (col 0) for col 1 is +1
srf[3] = 5      # SPM for C (col 0) for col 1 is +1
sim.setSPMLine(srf_spm_line, srf.copy())

# Prepare A
sim.setSPMLine(a_spm_line, matrix_A[:N_ELEMS_PER_VWR])
a_spm_line+=1
sim.setSPMLine(a_spm_line, matrix_A[N_ELEMS_PER_VWR:])

# Prepare B
matrix_B_reshaped = matrix_B.reshape(nColsA, nColsB) # Reshape into a 2D matrix
matrix_B_transposed = matrix_B_reshaped.T.flatten() # Transpose and flatten back into a 1D array
sim.setSPMLine(b_spm_line, matrix_B_transposed[:N_ELEMS_PER_VWR])
b_spm_line+=1
sim.setSPMLine(b_spm_line, matrix_B_transposed[N_ELEMS_PER_VWR:])

# Prepare C
# TODO: Fix this so it loads the blocks correctly, not C by rows
aux_c_line = c_spm_line
for ini in range(0,nRowsA*nColsB, N_ELEMS_PER_VWR):
    sim.setSPMLine(aux_c_line, matrix_C[ini:ini+N_ELEMS_PER_VWR])
    aux_c_line+=1

In [6]:
sim.displaySPMLine(0)
sim.displaySPMLine(1)
sim.displaySPMLine(2)
sim.displaySPMLine(3)
sim.displaySPMLine(4)

SPM 0: [16, 1, 3, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ]
SPM 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, ]
SPM 2: [128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 13

In [7]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

ASM to Hex
Processing file: ./kernels/mmul/instructions_asm_bb16x16_2col.csv...
Creating file: ./kernels/mmul/dsip_bitstream.h
Creating file: ./kernels/mmul/instructions_hex_bb16x16_2col_autogen.csv


Finally, we load the kernel into the internal memory of the specialized units and run it.

In [8]:
# --------------------------------------------
#                 LOAD KERNEL
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

# --------------------------------------------
#               SIMULATE EXECUTION
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

sim.run(kernel_number, display_ops=display_ops, max_iter=3000)

Processing file: ./kernels/mmul/instructions_hex_bb16x16_2col_autogen.csv...
---------------------
     PC[0]: 0
---------------------
LSU: NOP/LD.VWR SRF --> ALU res = 0
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: NOP (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 0
LCU: NOP --> ALU res = 0
---------------------
     PC[1]: 0
---------------------
LSU: NOP/LD.VWR SRF --> ALU res = 0
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: NOP (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 0
LCU: NOP --> ALU res = 0
---------------------
     PC[0]: 1
---------------------
LSU: SADD R7, ZERO, SRF(3)/NOP --> ALU res = 5
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: SADD R5, ZERO, LAST (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 31
LCU: NOP --> ALU res = 0
---------------------
     PC[1

We can check it more rigorously. We can define our function in python and check that the output matches the CGRA output.

In [9]:
def mmul (in_A, in_B, nRowsA, nColsA, nColsB):
    out = np.zeros(nRowsA*nColsB)
    for i in range(nRowsA):
        for j in range(nColsB):
            sum = 0
            for k in range(nColsA):
                sum += int(in_A[i*nColsA + k] * in_B[k*nColsB + j])
            out[i*nColsB + j] = sum
    return [int(elem) for elem in out]

In [10]:
# Get output from the CGRA
disco_cgra_res_0 = sim.getSPMLine(c_spm_line)


In [11]:
from itertools import groupby

def comprimir_rangos(arr):
    arr.sort()  # Asegurarse de que esté ordenado
    rangos = []
    
    for _, grupo in groupby(enumerate(arr), lambda x: x[1] - x[0]):
        grupo = [x[1] for x in grupo]  # Extraer los valores
        if len(grupo) > 1:
            rangos.append(f"{grupo[0]}-{grupo[-1]}")
        else:
            rangos.append(f"{grupo[0]}")

    return ", ".join(rangos)

def imprimir_por_linea(arr, tam_linea=8):
    for i in range(0, len(arr), tam_linea):
        print([int(x) for x in arr[i:i+tam_linea]])

In [12]:
# Prepare output
# Extraer bloques C0, C1, ..., C7 en orden correcto
nBloques = 16
tamBloque = 8
# Reorganizar los bloques en el orden correcto
array_ordenado = []
for i in range(nColsCGRA):
    ini = 2*tamBloque*i
    for j in range (nRCs):
        array_ordenado.extend(disco_cgra_res[ini:ini+2*tamBloque])
        ini += 4*tamBloque

In [13]:
errors_idx = []
expected_output = mmul(matrix_A, matrix_B, nRowsA, nColsA, nColsB)
for i in range(len(expected_output)):
    if expected_output[i] != array_ordenado[i]:
        errors_idx.append(i)
if len(errors_idx) == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(len(errors_idx)) + " errors.")
    print(comprimir_rangos(errors_idx))
    print("DISCO-CGRA result:")
    imprimir_por_linea(disco_cgra_res)
    print("DISCO-CGRA reordered:")
    imprimir_por_linea(array_ordenado)
    print("Expected result:")
    imprimir_por_linea(expected_output)

IndexError: list index out of range